In [ ]:
import contextily as cx
import geopandas as gpd
import pandas as pd

In [ ]:
aws_ts_df_filepath = "../data/interim/aws-ts-df.csv"
lcs_ts_df_filepath = "../data/interim/lcs-ts-df.csv"

aws_stations_gdf_filepath = "../data/interim/aws-stations.gpkg"
lcs_stations_gdf_filepath = "../data/interim/lcs-stations.gpkg"

stations_crs = "epsg:2056"
source_labels = ["AWS", "LCS"]

dst_station_id_col = "station_id"
dst_stations_gdf_filepath = "../data/processed/stations.gpkg"
dst_ts_df_filepath = "../data/processed/ts-df.csv"

In [ ]:
# merge both data sources: resample each to hourly and concatenate along columns
ts_df = pd.concat(
    [
        pd.concat(
            [
                heatwave_ts_df.loc[heatwave]
                .resample("h")
                .mean()
                .assign(**{"heatwave": heatwave})
                for heatwave, heatwave_ts_df in pd.read_csv(
                    ts_df_filepath,
                    index_col=["heatwave", "time"],
                    parse_dates=["time"],
                ).groupby("heatwave")
            ],
            axis="rows",
        )
        for ts_df_filepath in [aws_ts_df_filepath, lcs_ts_df_filepath]
    ],
    axis="columns",
)
# deduplicate heatwave column from the two sources, restore MultiIndex
ts_df = (
    ts_df.loc[:, ~ts_df.columns.duplicated()]
    .reset_index()
    .set_index(["heatwave", "time"])
)
ts_df

In [ ]:
stations_gdf = (
    pd.concat(
        [
            gpd.read_file(stations_gdf_filepath)
            .to_crs(stations_crs)
            .rename(columns={"index": dst_station_id_col})[
                [dst_station_id_col, "geometry"]
            ]
            .assign(source=source)
            for stations_gdf_filepath, source in zip(
                [aws_stations_gdf_filepath, lcs_stations_gdf_filepath],
                source_labels,
            )
        ]
    )
    .set_index(dst_station_id_col)
    .loc[ts_df.columns]
    .reset_index(names=dst_station_id_col)
)
ax = stations_gdf.plot("source", legend=True)
cx.add_basemap(ax, crs=stations_gdf.crs)

In [ ]:
stations_gdf["source"].value_counts()

In [ ]:
ts_df.to_csv(dst_ts_df_filepath)
stations_gdf.to_file(dst_stations_gdf_filepath)